# Quarterly Pro Forma Income Statement with PROC COMPUTAB

## Executive Summary

This notebook builds a regional bank's quarterly pro forma income statement directly from monthly ledger data using PROC COMPUTAB, the SAS/ETS report-writing table procedure. It routes each month's interest income, interest expense, fee income, and operating costs into the correct calendar-quarter column, then uses row programming blocks to compute net interest income, pre-tax income, and net income, and a column block to roll the four quarters into a full-year total.

## Data Sources

The single synthetic dataset `bank_ledger` simulates one fiscal year of monthly financial-statement line items for a mid-size community bank. Twelve monthly observations are generated inline with `call streaminit`/`rand` so the notebook is fully self-contained.

| Variable | Type | Description |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Month-end statement date (Jan–Dec) |
| `loanint`  | Num | Interest and fees earned on the loan portfolio (USD thousands) |
| `secint`   | Num | Interest earned on the investment securities book (USD thousands) |
| `depint`   | Num | Interest paid on deposits (USD thousands) |
| `borrint`  | Num | Interest paid on borrowings / FHLB advances (USD thousands) |
| `feeinc`   | Num | Non-interest (fee & service-charge) income (USD thousands) |
| `salaries` | Num | Salaries and employee-benefit expense (USD thousands) |
| `occupancy`| Num | Occupancy and equipment expense (USD thousands) |
| `othropex` | Num | Other non-interest operating expense (USD thousands) |
| `provision`| Num | Provision for credit losses (USD thousands) |
| `taxrate`  | Num | Effective tax rate applied to pre-tax income |

# Quarterly Pro Forma Income Statement with PROC COMPUTAB

Bank finance teams routinely turn a monthly general ledger into a **quarterly income statement** that shows net interest income and bottom-line net income. `PROC COMPUTAB` (SAS/ETS) is purpose-built for this: it lays out a programmable table whose **columns** are the reporting periods and whose **rows** are statement line items, and it lets you compute subtotals with ordinary DATA step logic in row and column blocks.

In this notebook we:

1. Generate one fiscal year of synthetic monthly ledger data for a community bank.
2. Route each month into its calendar quarter and build a quarterly income statement.
3. Compute net interest income, pre-tax income, and net income in a **row block**, and roll the quarters into a full-year total in a **column block**.
4. Reuse the `out=` table so the computed statement can feed downstream reporting.

## Step 1 — Generate synthetic monthly ledger data

We simulate twelve month-end observations. Loan and securities interest income trend gently upward over the year, deposit and borrowing costs scale with the rate environment, and fee income, operating expenses, and the credit-loss provision carry realistic month-to-month noise. `call streaminit` fixes the seed so the data is reproducible.

In [1]:
data bank_ledger;
   call streaminit(20240115);
   format stmtdate date9.;
   do month = 1 to 12;
      /* Month-end statement date for fiscal year 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Mild upward drift across the year + monthly noise */
      drift = 1 + 0.012 * (month - 1);

      /* Interest income (USD thousands) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Interest expense (USD thousands) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Non-interest income and expense (USD thousands) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Provision for credit losses, occasionally elevated */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Effective tax rate */
      taxrate = 0.21;

      output;
   end;
   keep stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
run;

proc print data=bank_ledger noobs label;
   title 'Synthetic Monthly Ledger — Community Bank FY2024';
   format loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
run;

                                    Synthetic Monthly Ledger — Community Bank FY2024                                    

 STMTDATE   LOANINT  SECINT  DEPINT  BORRINT  FEEINC  SALARIES  OCCUPANCY  OTHROPEX  PROVISION  TAXRATE
28JAN2024  1,772.95  423.79  531.47   128.99  339.33    699.38     171.95    202.43     130.93     0.21
28FEB2024  1,824.38  421.13  564.85   125.53  294.09    767.29     162.79    307.61     123.25     0.21
28MAR2024  1,931.98  442.06  536.64   131.71  295.72    724.03     153.26    254.21     115.76     0.21
28APR2024  1,934.99  439.29  535.94   140.01  294.56    729.47     174.19    237.43     198.05     0.21
28MAY2024  1,949.31  447.35  591.44   124.42  299.78    739.13     165.08    223.32     105.57     0.21
28JUN2024  1,934.36  448.28  551.70   147.64  335.81    718.90     171.91    236.94     130.13     0.21
28JUL2024  1,936.57  439.22  565.70   133.82  293.21    743.87     174.16    247.19     199.20     0.21
28AUG2024  1,979.73  457.42  558.54   144.45  

## Step 2 — Build the quarterly income statement

The heart of the report is the `PROC COMPUTAB` step below.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** defines four quarter columns plus a full-year column.
- The unlabeled **input block** sets the automatic variable **`_col_`** with `qtr(stmtdate)`, which routes each monthly observation into the right quarter column. Because the input is transposed by default, each ledger variable lands on the row that shares its name.
- The **row block** `inc_stmt:` runs once per column and computes the derived lines — net interest income, total non-interest expense, pre-tax income, and net income — exactly as an accountant would.
- The **column block** `total:` runs once per row and sums the four quarters into the `fy` (full-year) column.

The `rows ... / ...` statements add titles, indentation, and rule lines (`ol` overline, `ul` underline, `dul` double underline) so the output reads like a real financial statement.

In [2]:
title 'Pro Forma Income Statement — Community Bancorp, Inc.';
title2 'Fiscal Year 2024  (Amounts in USD Thousands)';

proc computab data=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Four quarters plus a rolled-up full-year column */
   columns qtr1 qtr2 qtr3 qtr4 fy / format=comma11.1;
   columns qtr1 / 'Q1';
   columns qtr2 / 'Q2';
   columns qtr3 / 'Q3';
   columns qtr4 / 'Q4';
   columns fy   / 'Full' 'Year' +3;

   /* Income statement rows, top to bottom */
   rows loanint  / 'Interest & Fees on Loans';
   rows secint   / 'Interest on Securities'        ul;
   rows intinc   / 'Total Interest Income';
   rows depint   / 'Interest on Deposits';
   rows borrint  / 'Interest on Borrowings'        ul;
   rows intexp   / 'Total Interest Expense';
   rows nii      / 'Net Interest Income'           ol skip;
   rows provision/ 'Provision for Credit Losses'   ul;
   rows niiap    / 'Net Int. Income after Prov.'   skip;
   rows feeinc   / 'Non-Interest Income'           ul;
   rows salaries / 'Salaries & Benefits';
   rows occupancy/ 'Occupancy & Equipment';
   rows othropex / 'Other Operating Expense'       ul;
   rows nonintexp/ 'Total Non-Interest Expense'    skip;
   rows pretax   / 'Pre-Tax Income'                ol;
   rows taxexp   / 'Income Tax Expense'            ul;
   rows netinc   / 'Net Income'                    dul;

   /* Input block: route each month to its calendar quarter */
   _col_ = qtr(stmtdate);

   /* Row block: compute statement subtotals across every column */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Column block: roll the four quarters into the full-year column */
   total:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
run;

                                  Pro Forma Income Statement — Community Bancorp, Inc.                                  
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


                             The COMPUTAB Procedure                             

                             Q1           Q2           Q3           Q4         Full  
                                                                               Year  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Interest & Fees on Loans
  loanint               5529.31      5818.66      5963.46      6276.96     23588.39  
  Interest on Securities
  secint                1286.98      1334.92      1342.03      1452.88      5416.81  
                    -----------  -----------  -----------  -----------  -----------  
  Total Interest Inc

## Step 3 — Reuse the COMPUTAB output dataset

The `PROC COMPUTAB` step above wrote its computed table to `qtr_income` via `out=`. Each row of that dataset is a statement line item and each column variable (`qtr1`–`qtr4`, `fy`) holds the computed value, so it can feed downstream reporting. Below we print the rolled-up full-year column to confirm the figures tie out.

In [3]:
title 'COMPUTAB Output Dataset — Full-Year Column';

proc print data=qtr_income label noobs;
   var _row_ fy;
   format fy comma12.1;
   label _row_ = 'Line Item' fy = 'Full Year';
run;

title;

                                       COMPUTAB Output Dataset — Full-Year Column                                       
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


Line Item  Full Year
---------  ---------
loanint     23,588.4
secint       5,416.8
intinc      29,005.2
depint       6,831.2
borrint      1,650.7
intexp       8,482.0
nii         20,523.2
provision    1,568.9
niiap       18,954.3
feeinc       3,703.2
salaries     8,763.1
occupancy    1,985.6
othropex     2,944.8
nonintexp   13,693.5
pretax       8,964.1
taxexp       1,882.5
netinc       7,081.6

NOTE: Option TITLE changed to COMPUTAB Output Dataset — Full-Year Column.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## Interpreting the results

The pro forma statement reads top to bottom like a bank's regulatory call report: total interest income less total interest expense yields **net interest income** — here about \$20.5 million for the year — the bank's primary earnings driver. Subtracting the **provision for credit losses**, adding **fee income**, and netting out **non-interest expense** gives pre-tax income of roughly \$9.0 million, and applying the 21% effective tax rate produces **net income** near \$7.1 million. The `total:` column block adds the four quarters into the full-year column, so the annual totals reconcile to the sum of the quarters by construction — confirmed in the `out=` dataset, where the full-year `netinc` of 7,081.6 equals the sum of the four quarterly figures.

Because everything is computed inside `PROC COMPUTAB`'s programmable row and column blocks, swapping in a real monthly ledger requires no change to the report logic — only the input dataset changes. The `out=` dataset then makes the computed statement available for dashboards, trend analysis, or board-package automation.